# Contexto do Problema

Você recebeu duas bases de dados de um comércio eletrônico:vendas: histórico de transações com valores, categorias, datas e status. clientes: cadastro com o identificador do cliente, nome e cidade. Seu objetivo como analista é tratar os dados ausentes, cruzar as bases, extrair métricas de negócio e preparar a base para relatórios gerenciais. Dados para Configuração do Ambiente. Execute o bloco abaixo para criar os DataFrames iniciais para o exercício:


### Importando libs


In [4]:
import numpy as np
import pandas as pd

### Base de dados


In [5]:

# Base de vendas
dados_vendas = {
    'cliente_id': [101, 102, 103, 101, 104, 102, 105, 103],
    'valor': [3500.75, 189.50, np.nan, 1200.00, 450.00, np.nan, 89.90, 780.50],
    'categoria': [
        'Eletronicos',
        'Livros',
        'Roupas',
        'Eletronicos',
        'Automotivo',
        'Livros',
        'Roupas',
        'Roupas',
    ],
    'data_hora': [
        '2024-01-15 10:23:00',
        '2024-01-18 14:05:00',
        '2024-02-05 09:12:00',
        '2024-02-20 16:40:00',
        '2024-03-02 11:00:00',
        '2024-03-15 18:30:00',
        '2024-04-10 08:20:00',
        '2024-04-22 13:45:00',
    ],
    'status': [
        'Concluído',
        'Concluído',
        'Pendente',
        'Concluído',
        'Cancelado',
        'Concluído',
        'Concluído',
        'Concluído',
    ],
    'email': [
        'maria@gmail.com',
        'joao@outlook.com',
        'ana@yahoo.com',
        'maria@gmail.com',
        'carlos@gmail.com',
        'joao@outlook.com',
        'lucas@empresa.com.br',
        'ana@yahoo.com',
    ],
}

# Base de clientes
dados_clientes = {
    'cliente_id': [101, 102, 103, 104, 105],
    'nome': [
        'Maria Silva',
        'Joao Souza',
        'Ana Oliveira',
        'Carlos Lima',
        'Lucas Mendes',
    ],
    'cidade': [
        'Sao Paulo',
        'Rio de Janeiro',
        'Belo Horizonte',
        'Curitiba',
        'Salvador',
    ],
}

df_vendas = pd.DataFrame(dados_vendas)
df_clientes = pd.DataFrame(dados_clientes)



## Parte 1: Diagnóstico e Limpeza

### 1. Dimensões e tipos de dados


In [6]:
# Dimensões do DataFrame
print(df_vendas.shape)

# Resumo dos tipos de dados e valores não nulos
df_vendas.info()

(8, 6)
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cliente_id  8 non-null      int64  
 1   valor       6 non-null      float64
 2   categoria   8 non-null      str    
 3   data_hora   8 non-null      str    
 4   status      8 non-null      str    
 5   email       8 non-null      str    
dtypes: float64(1), int64(1), str(4)
memory usage: 516.0 bytes


### 2. Valores nulos e preenchimento com a mediana


In [7]:
# Identificando os valores nulos
print(df_vendas.isnull().sum())

# Preenche com a mediana da categoria; se a categoria não tiver mediana, usa a mediana global
mediana_por_categoria = df_vendas.groupby('categoria')['valor'].transform('median')
df_vendas['valor'] = df_vendas['valor'].fillna(mediana_por_categoria)
df_vendas['valor'] = df_vendas['valor'].fillna(df_vendas['valor'].median())

df_vendas

cliente_id    0
valor         2
categoria     0
data_hora     0
status        0
email         0
dtype: int64


,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com


### 3. Transações Concluídas com valor superior a R$ 500,00


In [8]:
# Usando .loc[]
vendas_concluidas_500 = df_vendas.loc[
    (df_vendas['status'] == 'Concluído') & (df_vendas['valor'] > 500)
]

# Alternativa equivalente usando .query()
# vendas_concluidas_500 = df_vendas.query("status == 'Concluído' and valor > 500")

vendas_concluidas_500

,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com


## Parte 2: Cruzamento e Transformação

### 4. Junção relacional (left merge) entre df_vendas e df_clientes


In [9]:
df_vendas_completo = df_vendas.merge(df_clientes, on='cliente_id', how='left')
df_vendas_completo

,cliente_id,valor,categoria,data_hora,status,email,nome,cidade
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte


### 5. Coluna `media_categoria` com `.groupby()` + `.transform()`


In [10]:
df_vendas_completo['media_categoria'] = df_vendas_completo.groupby('categoria')['valor'].transform('mean')
df_vendas_completo

,cliente_id,valor,categoria,data_hora,status,email,nome,cidade,media_categoria
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba,450.000
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador,435.200
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200


### 6. Clientes que usam e-mail @gmail.com


In [11]:
clientes_unicos = df_vendas_completo.drop_duplicates('cliente_id')

usa_gmail = clientes_unicos['email'].str.contains('@gmail.com')
qtd_clientes_gmail = usa_gmail.sum()

print(f'Clientes com e-mail @gmail.com: {qtd_clientes_gmail}')
clientes_unicos.loc[usa_gmail, ['cliente_id', 'nome', 'email']]

Clientes com e-mail @gmail.com: 2


,cliente_id,nome,email
0,101,Maria Silva,maria@gmail.com
4,104,Carlos Lima,carlos@gmail.com


## Parte 3: Análise Temporal e Agregação

### 7. Conversão para datetime e extração de mês/dia da semana


In [12]:
df_vendas_completo['data_hora'] = pd.to_datetime(df_vendas_completo['data_hora'])

df_vendas_completo['mes'] = df_vendas_completo['data_hora'].dt.month_name()
df_vendas_completo['dia_semana'] = df_vendas_completo['data_hora'].dt.day_name()

df_vendas_completo[['data_hora', 'mes', 'dia_semana']]

,data_hora,mes,dia_semana
0,2024-01-15 10:23:00,January,Monday
1,2024-01-18 14:05:00,January,Thursday
2,2024-02-05 09:12:00,February,Monday
3,2024-02-20 16:40:00,February,Tuesday
4,2024-03-02 11:00:00,March,Saturday
5,2024-03-15 18:30:00,March,Friday
6,2024-04-10 08:20:00,April,Wednesday
7,2024-04-22 13:45:00,April,Monday


### 8. Tabela dinâmica: total de vendas por categoria x cidade


In [13]:
tabela_dinamica = pd.pivot_table(
    df_vendas_completo,
    values='valor',
    index='categoria',
    columns='cidade',
    aggfunc='sum',
    fill_value=0,
)

tabela_dinamica

cidade,Belo Horizonte,Curitiba,Rio de Janeiro,Salvador,Sao Paulo
categoria,,,,,
Automotivo,0.0,450.0,0.0,0.0,0.00
Eletronicos,0.0,0.0,0.0,0.0,4700.75
Livros,0.0,0.0,379.0,0.0,0.00
Roupas,1215.7,0.0,0.0,89.9,0.00
